In [1]:
import os


In [2]:
%pwd

'd:\\AI_project\\Deep_Learning\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\AI_project\\Deep_Learning'

#### Steps:
1. Updateconfig.yaml
2. params.yaml (optional)
3. update entity
4. update configuration manager
5. update components
6. pipeline
7. main.py

#### Entity :


In [10]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class TrainingConfig:
    root_dir : Path
    trained_model_path : Path
    updated_base_model_path : Path
    training_data : Path
    params_epochs : int
    params_batch_size : int
    params_is_augmentation :bool
    params_image_size : list

#### Configuration Manager :


In [11]:
from src.cnnClassifier.constants import *
from src.cnnClassifier.utils.common import read_yaml,create_directories

class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_training_config(self) -> TrainingConfig:
        training = self.config.training
        prepare_base_model = self.config.prepare_base_model
        params = self.params
        training_data = os.path.join(self.config.data_ingestion.unzip_dir, "kidney-ct-scan-image")
        create_directories([
            Path(training.root_dir)
        ])

        training_config = TrainingConfig(
            root_dir=Path(training.root_dir),
            trained_model_path=Path(training.trained_model_path),
            updated_base_model_path=Path(prepare_base_model.updated_base_model_path),
            training_data=Path(training_data),
            params_epochs=params.EPOCHS,
            params_batch_size=params.BATCH_SIZE,
            params_is_augmentation=params.AUGMENTATION,
            params_image_size=params.IMAGE_SIZE
        )

        return training_config

   


#### Components :


In [12]:
import os
import urllib.request as request
from zipfile import ZipFile
import tensorflow as tf
import time
import tensorflow as tf




class Training:
    def __init__(self, config: TrainingConfig):
        self.config = config


   
    def get_base_model(self):
        self.model = tf.keras.models.load_model(   # For loading the model
            self.config.updated_base_model_path
        )
        self.model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
                           loss='categorical_crossentropy',  # Or your actual loss function
                           metrics=['accuracy']
)


    def train_valid_generator(self):


        datagenerator_kwargs = dict(
            rescale = 1./255, # 1. is a float value
            validation_split=0.20
        )


        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],  # It will remove 3 from the params.yaml
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )


        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )


        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )


        if self.config.params_is_augmentation:   # It will check whether the params file augmetation is True if than it will excute if block 
            train_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
                rotation_range=40,
                horizontal_flip=True,
                width_shift_range=0.2,
                height_shift_range=0.2,
                shear_range=0.2,
                zoom_range=0.2,
                **datagenerator_kwargs
            )
        else:
            train_datagenerator = valid_datagenerator


        self.train_generator = train_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="training",
            shuffle=True,
            **dataflow_kwargs
        )


   
    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)






   
    def train(self):
        self.steps_per_epoch = self.train_generator.samples // self.train_generator.batch_size   # 60000 / 16 = 6000 it will this much only images
        self.validation_steps = self.valid_generator.samples // self.valid_generator.batch_size


        self.model.fit(
            self.train_generator,
            epochs=self.config.params_epochs,
            steps_per_epoch=self.steps_per_epoch,
            validation_steps=self.validation_steps,
            validation_data=self.valid_generator
        )


        self.save_model(  # Here path for saving the model
            path=self.config.trained_model_path,
            model=self.model
        )


In [ ]:
#Pipeline


try:
    config = ConfigurationManager()
    training_config = config.get_training_config()
    training = Training(config=training_config)
    training.get_base_model()
    training.train_valid_generator()
    training.train()
    
except Exception as e:
    raise e



[2026-01-08 12:02:52,544: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-01-08 12:02:52,547: INFO: common: yaml file: params.yaml loaded successfully]
[2026-01-08 12:02:52,550: INFO: common: created directory at: artifacts]
[2026-01-08 12:02:52,550: INFO: common: created directory at: artifacts\training]
Found 93 images belonging to 2 classes.
Found 372 images belonging to 2 classes.
23/23 ━━━━━━━━━━━━━━━━━━━━ 80s 3s/step - accuracy: 0.7880 - loss: 0.6564 - val_accuracy: 0.9625 - val_loss: 0.1386
